# Project8 — huấn luyện tiếp trên Kaggle Notebooks

Tải ZIP lên Kaggle như một **Dataset**, rồi gắn dataset đó vào Notebook. Chọn accelerator CPU/GPU trong Notebook Settings. Kaggle lưu output của version đã chạy thành công; file đang nằm trong session có thể mất nếu session bị ngắt trước khi lưu version hoặc tải về. Không chọn GPU mặc định nếu chưa có quota/accelerator khả dụng. Test split không có trong ZIP và không dùng để chọn model/tham số.


In [ ]:
# Cấu hình người dùng
BUNDLE_INPUT = '/kaggle/input'  # tự tìm bundle khi chỉ gắn một Dataset; nếu nhiều bundle, đặt đường dẫn Dataset cụ thể
EXECUTION_DEVICE = 'GPU'  # 'GPU' hoặc 'CPU'; chỉnh Notebook Settings > Accelerator
START_TRAINING = False  # chỉ bật sau khi các kiểm tra bên dưới đạt
RUN_MODE = 'NEW_FINE_TUNE'  # 'NEW_FINE_TUNE' hoặc 'RESUME_KAGGLE'
RESUME_CHECKPOINT = ''  # .../kaggle_runs/<run_id>/weights/last.pt từ output đã lưu
EPOCHS = 5
IMGSZ = 768
BATCH = 4  # CPU thử 2; nếu OOM, giảm batch hoặc imgsz trong run mới
CLOSE_MOSAIC = 2  # augmentation ghép hoạt động 3 epoch đầu, tắt 2 epoch cuối
OUTPUT_ROOT = '/kaggle/working/BTL-AI-Camera'

if EXECUTION_DEVICE not in {'GPU', 'CPU'}:
    raise ValueError('EXECUTION_DEVICE chỉ nhận GPU hoặc CPU')
if RUN_MODE not in {'NEW_FINE_TUNE', 'RESUME_KAGGLE'}:
    raise ValueError('RUN_MODE chỉ nhận NEW_FINE_TUNE hoặc RESUME_KAGGLE')


### 1. Gắn Dataset và stage bundle

Tải ZIP private lên Kaggle như Dataset, rồi trong Notebook chọn **Add Input** để gắn Dataset. Kaggle tự bung ZIP khi upload Dataset; cell hỗ trợ cả cây thư mục đã bung có `FILES.json` lẫn ZIP còn nguyên dưới `/kaggle/input`. Thay slug ở `BUNDLE_INPUT` nếu cần. Nội dung được sao chép vào `/tmp` scratch và kiểm SHA-256; đầu ra model chỉ ghi vào `/kaggle/working`.


In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path

def file_sha(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def stage_bundle(input_root, work):
    input_root, work = Path(input_root), Path(work)
    if work.exists(): shutil.rmtree(work)
    work.mkdir(parents=True)
    roots = [path.parent for path in input_root.rglob('FILES.json')]
    if len(roots) == 1:
        bundle = work / 'project8_kaggle'
        shutil.copytree(roots[0], bundle)
    elif len(roots) > 1:
        raise RuntimeError(f'Tìm thấy nhiều FILES.json dưới {input_root}; đặt BUNDLE_INPUT sát Dataset cần dùng.')
    else:
        archives = list(input_root.rglob('project8_kaggle_bundle.zip'))
        if len(archives) != 1:
            raise FileNotFoundError(f'Cần cây bundle đã bung hoặc một project8_kaggle_bundle.zip trong {input_root}')
        local_zip = work / 'project8_kaggle_bundle.zip'
        shutil.copy2(archives[0], local_zip)
        if file_sha(archives[0]) != file_sha(local_zip):
            raise RuntimeError('ZIP nguồn và bản copy scratch không khớp SHA-256')
        with zipfile.ZipFile(local_zip) as archive:
            root = work.resolve()
            for member in archive.infolist():
                target = (work / member.filename).resolve()
                if root not in target.parents and target != root:
                    raise RuntimeError(f'Đường dẫn ZIP không an toàn: {member.filename}')
            archive.extractall(work)
        bundle = work / 'project8_kaggle'
    files = json.loads((bundle / 'FILES.json').read_text(encoding='utf-8'))
    for rel, record in files['files'].items():
        path = bundle / rel
        if not path.is_file() or path.stat().st_size != record['bytes'] or file_sha(path) != record['sha256']:
            raise RuntimeError(f'Kiểm tra file thất bại: {rel}')
    return bundle, files

BUNDLE, FILES = stage_bundle(BUNDLE_INPUT, '/tmp/project8_kaggle_runtime')
print('Đã stage và xác minh', len(FILES['files']), 'file; split:', FILES['images'])


In [ ]:
# Chọn CPU/GPU theo accelerator Kaggle; yêu cầu GPU nhưng CUDA vắng mặt thì dừng rõ.
import subprocess, sys
import torch

def resolve_device(requested, cuda_available):
    if requested == 'GPU' and not cuda_available:
        raise RuntimeError('Đã chọn GPU nhưng CUDA không khả dụng. Chọn GPU trong Notebook Settings hoặc đổi cấu hình sang CPU.')
    return '0' if requested == 'GPU' else 'cpu'

if EXECUTION_DEVICE == 'GPU':
    subprocess.run(['nvidia-smi'], check=True)
DEVICE = resolve_device(EXECUTION_DEVICE, torch.cuda.is_available())
torch_before = (torch.__version__, torch.version.cuda, torch.cuda.is_available())
print('Python:', sys.version.split()[0], '| torch:', torch_before, '| device:', DEVICE)


In [ ]:
# Cài Ultralytics, không cài requirements CPU local lên Kaggle.
%pip install -q -r /tmp/project8_kaggle_runtime/project8_kaggle/requirements-kaggle.txt
import torch, yaml
torch_after = (torch.__version__, torch.version.cuda, torch.cuda.is_available())
print('torch sau cài đặt:', torch_after)
if torch_before[1:] != torch_after[1:] or (EXECUTION_DEVICE == 'GPU' and not torch.cuda.is_available()):
    raise RuntimeError('Cài dependency làm thay đổi CUDA; dừng và chọn lại môi trường Kaggle tương thích.')
from ultralytics import YOLO
DATA_DIR = BUNDLE / 'dataset'
DATA_YAML = DATA_DIR / 'runtime_data.yaml'
CONFIG = json.loads((BUNDLE / 'FILES.json').read_text(encoding='utf-8'))
assert list(yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))['names'].values()) == CONFIG['classes']
initial = BUNDLE / 'checkpoints/initial_project8.pt'
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and list(model_check.names.values()) == CONFIG['classes']
print('Checkpoint project8 đã nạp đúng thứ tự 8 lớp.')


In [ ]:
# Smoke check: xác minh manifest và đường dẫn mà trainer thực sự sử dụng; không train/full val.
import csv, yaml
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

source_yaml = DATA_DIR / 'runtime_data.yaml'
config = yaml.safe_load(source_yaml.read_text(encoding='utf-8'))
assert not config.get('test'), 'Không đưa test vào YAML dùng chọn model.'
with (DATA_DIR / 'export_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    manifest_rows = list(csv.DictReader(stream))
expected_counts = CONFIG['images']
assert set(expected_counts) == {'train', 'val'}
actual_counts = {split: sum(row['split'] == split for row in manifest_rows) for split in expected_counts}
assert actual_counts == expected_counts and len(manifest_rows) == sum(expected_counts.values())
for split in expected_counts:
    sample = next(row for row in manifest_rows if row['split'] == split)
    assert (DATA_DIR / sample['image_path']).is_file() and (DATA_DIR / sample['label_path']).is_file()

# path='.' được Ultralytics phân giải từ cwd, không phải thư mục YAML.
# Tạo YAML riêng ngoài cây bundle đã hash, với root tuyệt đối của bản staged.
config['path'] = str(DATA_DIR.resolve())
DATA_YAML = BUNDLE.parent / 'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
resolved_data = check_det_dataset(str(DATA_YAML), autodownload=False)
for split in expected_counts:
    assert Path(resolved_data[split]) == (DATA_DIR / 'images' / split).resolve()
print('Smoke check đạt:', actual_counts, '| trainer đọc đúng bản staged; không có test media.')


In [ ]:
# Training guard. Checkpoint và log được ghi từng epoch vào /kaggle/working.
if not START_TRAINING:
    print('Chưa train. Kiểm tra device/quota rồi đặt START_TRAINING=True khi sẵn sàng.')
else:
    # Kiểm lại ngay trước train để không dùng nhầm DEVICE cũ trong kernel.
    if EXECUTION_DEVICE != 'GPU' or not isinstance(DEVICE, str) or not DEVICE or any(not part.isdigit() for part in DEVICE.split(',')):
        raise RuntimeError('Chỉ cho phép train trên GPU CUDA được chỉ định rõ; không fallback sang CPU.')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA không khả dụng: dừng train, không fallback sang CPU.')
    gpu_ids = [int(part) for part in DEVICE.split(',')]
    if len(set(gpu_ids)) != len(gpu_ids) or any(gpu_id >= torch.cuda.device_count() for gpu_id in gpu_ids):
        raise RuntimeError(f'Không đủ GPU CUDA hoặc ID GPU bị trùng cho device={DEVICE}; dừng train.')
    print('Train bắt buộc dùng GPU:', DEVICE, '| CUDA khả dụng:', True, flush=True)
    import datetime, shutil
    from pathlib import Path
    RUNS = Path(OUTPUT_ROOT) / 'kaggle_runs'
    RUNS.mkdir(parents=True, exist_ok=True)
    if RUN_MODE == 'NEW_FINE_TUNE':
        run_name = datetime.datetime.now().strftime('project8_26n_kaggle_%Y%m%d_%H%M%S')
        model = YOLO(str(initial))  # optimizer/scheduler được khởi tạo lại
        model.train(data=str(DATA_YAML), project=str(RUNS), name=run_name, exist_ok=False,
                    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=2, cache=False,
                    fraction=1.0, optimizer='MuSGD', lr0=0.001, patience=10, seed=42,
                    mosaic=1.0, mixup=0.0, cutmix=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
                    close_mosaic=CLOSE_MOSAIC, box=7.5, cls=0.5, dfl=1.5, nms=False,
                    save=True, save_period=1, resume=False, val=True)
        RUN_DIR = RUNS / run_name
    else:
        last = Path(RESUME_CHECKPOINT)
        if not last.is_file() or last.name != 'last.pt':
            raise FileNotFoundError('RESUME_KAGGLE cần last.pt còn optimizer trong output đã lưu.')
        RUN_DIR = last.parents[1]
        model = YOLO(str(last))
        model.train(resume=True, device=DEVICE, workers=2, cache=False)
    best = RUN_DIR / 'weights/best.pt'
    last = RUN_DIR / 'weights/last.pt'
    if not best.is_file() or not last.is_file():
        raise RuntimeError('Thiếu best.pt hoặc last.pt; giữ artefact trong /kaggle/working để kiểm tra.')
    print('Run output:', RUN_DIR, '| best SHA-256:', file_sha(best), '| last:', file_sha(last))


In [ ]:
# Baseline COCO80 và fine-tuned cùng protocol bbox/NMS trên validation; không chấm test.
if START_TRAINING:
    import subprocess
    evaluator = BUNDLE / 'src/classroom_ai/experiments/evaluate.py'
    baseline = BUNDLE / 'checkpoints/baseline_yolo26n_coco80.pt'
    baseline_out = RUN_DIR / 'baseline_val_metrics.json'
    tuned_out = RUN_DIR / 'finetuned_val_metrics.json'
    common = [sys.executable, str(evaluator), '--manifest', str(DATA_DIR / 'export_manifest.csv'), '--device', DEVICE]
    subprocess.run(common + ['--weights', str(baseline), '--output', str(baseline_out)], check=True)
    subprocess.run(common + ['--weights', str(best), '--output', str(tuned_out)], check=True)
    print('Chấm 640/conf=.001/IoU=.70/NMS/max_det=300 trên val; metric AP từng lớp nằm trong JSON.')


In [ ]:
# Tạo result ZIP gồm checkpoint/hash/protocol/metrics để tải về và nhập local.
if START_TRAINING:
    import importlib.metadata, platform
    RESULT_DIR = RUN_DIR / 'kaggle_result'
    RESULT_DIR.mkdir(exist_ok=False)
    shutil.copy2(best, RESULT_DIR / 'best.pt')
    for source, name in ((baseline_out, 'baseline_val_metrics.json'), (tuned_out, 'finetuned_val_metrics.json'),
                         (RUN_DIR / 'results.csv', 'results.csv'), (RUN_DIR / 'args.yaml', 'args.yaml')):
        if Path(source).is_file(): shutil.copy2(source, RESULT_DIR / name)
    result_record = {
        'schema_version': 1, 'status': 'completed', 'run_id': RUN_DIR.name,
        'checkpoint_file': 'best.pt', 'checkpoint_sha256': file_sha(RESULT_DIR / 'best.pt'),
        'source_checkpoint_sha256': FILES['files']['checkpoints/initial_project8.pt']['sha256'],
        'release_id': FILES['source_release'], 'source_manifest_sha256': FILES['source_manifest_sha256'],
        'device': DEVICE, 'torch': torch.__version__, 'torch_cuda': torch.version.cuda,
        'ultralytics': importlib.metadata.version('ultralytics'), 'python': platform.python_version(),
        'epochs_requested': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'close_mosaic': CLOSE_MOSAIC,
        'metrics_protocol': json.loads(tuned_out.read_text(encoding='utf-8'))['protocol'],
        'metrics': json.loads(tuned_out.read_text(encoding='utf-8')),
    }
    (RESULT_DIR / 'result.json').write_text(json.dumps(result_record, indent=2, ensure_ascii=False), encoding='utf-8')
    result_files = {p.name: {'sha256': file_sha(p), 'bytes': p.stat().st_size}
                    for p in RESULT_DIR.iterdir() if p.is_file()}
    (RESULT_DIR / 'FILES.json').write_text(json.dumps({'files': result_files}, indent=2), encoding='utf-8')
    RESULT_ZIP = RUN_DIR / f'{RUN_DIR.name}_result.zip'
    with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3, allowZip64=True) as archive:
        for path in sorted(RESULT_DIR.iterdir()):
            if path.is_file(): archive.write(path, path.name)
    print('Result ZIP (download or Save Version output):', RESULT_ZIP, '| SHA-256:', file_sha(RESULT_ZIP))


## Lưu, phục hồi, hạn mức

Sau khi notebook và cấu hình đúng, dùng **Save Version → Save & Run All**. Kaggle docs hiện ghi session CPU/GPU tối đa 12 giờ và tối đa 20 GB output lưu trong `/kaggle/working`; `Save & Run All` chạy notebook sạch từ đầu và output được gắn với version thành công. Vì vậy kiểm quota trước, giới hạn lượt train ở 5 epoch, lưu checkpoint `last.pt` mỗi epoch, rồi tải result ZIP/checkpoint hoặc Save Version trước khi đóng. Nếu session ngắt trước khi lưu/tải, file chỉ trong session không được bảo đảm.

GPU quota được tài liệu Kaggle mô tả khoảng 30 giờ/tuần, đôi khi cao hơn tùy nhu cầu/tài nguyên; kiểm tra quota tài khoản. Không có GPU cụ thể hoặc tốc độ được bảo đảm.

`RESUME_KAGGLE` dùng khi `last.pt` còn optimizer và đã được lưu trong Kaggle output/version. `best.pt` sau khi hoàn tất thường đã strip optimizer, phù hợp làm nguồn cho run fine-tune mới. OOM: giữ artefact cũ, tạo run mới từ best với `BATCH=2` hoặc `IMGSZ=640`, ghi điều kiện khác. Không đổi cấu hình giữa chừng khi resume.

Lượt gợi ý sau R3: một run mới tối đa 5 epoch từ R3 best đã kiểm, 768/batch 4 nếu accelerator đủ bộ nhớ, `close_mosaic=2` để augmentation hoạt động 3 epoch đầu. Chưa đo Kaggle GPU thật. Baseline pretrained và fine-tuned dùng evaluator NMS chung trên val; test luôn ngoài chọn model/tham số.
